In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Yeast


# Databases Having Gene gene relation of yeast 

In [3]:
# Monarch/Monarch_final/Yeast/Gene_Yeast_MolecularFunction.csv

In [4]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Yeast_gene_molecularfunction
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Yeast/Yeast_gene_molecularfunction/Yeast_gene_molecularfunction.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name'
]

mkdir: cannot create directory ‘Yeast_gene_molecularfunction’: File exists


In [5]:
Monarch_gene_molecularfun = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Yeast/Gene_Yeast_MolecularFunction.csv')
Monarch_gene_molecularfun.columns = Monarch_gene_molecularfun.columns.str.lower()
Monarch_gene_molecularfun['head_id_is'] = 'SGD_SysematicName'
Monarch_gene_molecularfun['species'] = 'S.cerevisae'
Monarch_gene_molecularfun['kg_type'] = 'Generalised'
Monarch_gene_molecularfun['tail_id_is'] = 'Quick_GO'
Monarch_gene_molecularfun['kg_source'] = 'Monarch_kg'

print(f"Monarch_gene_molecularfun: {len(Monarch_gene_molecularfun):,} rows")
Monarch_gene_molecularfun.head(3)


Monarch_gene_molecularfun: 37,189 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_name,species,kg_type,kg_source
0,YNL032W,Gene_MolecularFunction,GO:0052845,Gene,enables,MolecularFunction,infores:sgd,OCA3|putative tyrosine protein phosphatase SIW14,"inositol-5-diphosphate-1,2,3,4,6-pentakisphosp...",Saccharomyces cerevisiae S288C,NaN,SGD_SysematicName,Quick_GO,SIW14,S.cerevisae,Generalised,Monarch_kg
1,YGR178C,Gene_MolecularFunction,GO:0044877,Gene,enables,MolecularFunction,infores:sgd,MRS16,protein-containing complex binding,Saccharomyces cerevisiae S288C,NaN,SGD_SysematicName,Quick_GO,PBP1,S.cerevisae,Generalised,Monarch_kg
2,YDL110C,Gene_MolecularFunction,GO:0098772,Gene,enables,MolecularFunction,infores:sgd,ADC17,molecular function regulator activity,Saccharomyces cerevisiae S288C,NaN,SGD_SysematicName,Quick_GO,TMA17,S.cerevisae,Generalised,Monarch_kg


In [6]:
# List all source DataFrames to include
source_dfs = [
    Monarch_gene_molecularfun
    # ← add further source DataFrames here
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 37,189


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,YNL032W,Gene_MolecularFunction,GO:0052845,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,OCA3|putative tyrosine protein phosphatase SIW14,"inositol-5-diphosphate-1,2,3,4,6-pentakisphosp..."
1,YGR178C,Gene_MolecularFunction,GO:0044877,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,MRS16,protein-containing complex binding
2,YDL110C,Gene_MolecularFunction,GO:0098772,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,ADC17,molecular function regulator activity
3,YDR291W,Gene_MolecularFunction,GO:0043138,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,ATP-dependent 3'-5' DNA helicase,3'-5' DNA helicase activity
4,YDR291W,Gene_MolecularFunction,GO:0042162,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,ATP-dependent 3'-5' DNA helicase,telomeric DNA binding
...,...,...,...,...,...,...,...,...,...,...,...,...
37184,YOR008C,Gene_MolecularFunction,GO:0004888,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,HCS77|WSC1,transmembrane signaling receptor activity
37185,YMR054W,Gene_MolecularFunction,GO:0051117,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,H(+)-transporting V0 sector ATPase subunit a,ATPase binding
37186,YOR210W,Gene_MolecularFunction,GO:0008270,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,ABC10-beta|DNA-directed RNA polymerase core su...,zinc ion binding
37187,YDL083C,Gene_MolecularFunction,GO:0003723,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,40S ribosomal protein uS9 RPS16B|rp61R|S16B|S9...,RNA binding


# Sanity Check — Distinct Values

In [7]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Gene_MolecularFunction'}
head_type           : {'Gene'}
tail_type           : {'MolecularFunction'}
relation_type       : {'enables', 'contributes_to'}
kg_source           : {'Monarch_kg'}
head_id_is          : {'SGD_SysematicName'}
tail_id_is          : {'Quick_GO'}


In [8]:

# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 37,189 remaining


In [9]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [10]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 37,189  |  After dedup: 21,203


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,15S_RRNA,Gene_MolecularFunction,GO:0003735,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,15S_RRNA,structural constituent of ribosome
1,21S_RRNA,Gene_MolecularFunction,GO:0003735,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,21S_RRNA,structural constituent of ribosome
2,AAD6,Gene_MolecularFunction,GO:0016491,Gene,enables,MolecularFunction,Monarch_kg,Generalised,SGD_SysematicName,Quick_GO,AAD6,oxidoreductase activity


In [11]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [12]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'Monarch_kg'} kg_source
Monarch_kg    21203
Name: count, dtype: int64


In [13]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    21203
Name: count, dtype: int64


In [14]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 21,203 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Yeast/Yeast_gene_molecularfunction/Yeast_gene_molecularfunction.csv


In [15]:
#